# IncrementalEdit on Colab

First real GPU execution of `NewWork/IncrementalEdit` (built without a GPU/torch available — see the project's `README.md` "What's verified vs. what isn't" section). The one piece most likely to need a fix on first run is `kontext_injection.py`'s `assert_reference_slice()`, which guesses the diffusers method name `pipe._encode_vae_image` for the Stage 2 check — if that Stage 2 line prints `[stage2] assertion could not run: ...`, the edit still proceeds (Stage 2 is diagnostic, not a hard stop), but copy the error text back for a fix.

Runtime > Change runtime type > GPU (A100/L4/T4 with >=24GB VRAM — an A100 is safest for 1024px Kontext).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

Edit `REPO_URL` if needed, or skip this cell and upload/mount `NewWork/IncrementalEdit` yourself — either way, end up with the folder at `/content/repo/NewWork/IncrementalEdit`.

In [ ]:
REPO_URL = "https://github.com/ashfaqfardin/cherry_on_top_exp.git"
!git clone --depth 1 "$REPO_URL" /content/repo
%cd /content/repo/NewWork/IncrementalEdit

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python -c "import ast; [ast.parse(open(f, encoding='utf-8').read()) for f in ['manifest.py','mask_ops.py','metrics.py','kontext_injection.py','run_incremental_edit.py']]; print('all files parse clean')"
!python test_cpu.py

## 3. Hugging Face auth

Both `FLUX.1-dev` and `FLUX.1-Kontext-dev` are gated — accept each model's license on huggingface.co first (while logged in as the account whose token you use here), or every `pipeline.from_pretrained(...)` call below will 401.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Stage 1 — build the canvas (`FLUX.1-dev`)

Prints the Stage 1 determinism VERDICT (`latent MSE(runA,runB)` — must be exactly `0.0`, per `pipelineInc.md` §5/§10).

In [ ]:
PROJECT = "runs/driveway"
!python run_incremental_edit.py init \
    --project-dir {PROJECT} \
    --prompt "an empty driveway at dusk, photorealistic" \
    --seed 42 --steps 28

In [ ]:
from PIL import Image
import json, glob

def show_latest(project=PROJECT):
    with open(f"{project}/manifest.json") as f:
        m = json.load(f)
    rev = m["revisions"][-1]
    print(f"revision {rev['id']}  op={rev['op']}  prompt={rev['prompt']!r}")
    return Image.open(f"{project}/{rev['image']}")

show_latest()

## 5. `add` — insert an object (`FLUX.1-Kontext-dev`)

This is the first call that exercises `kontext_injection.py` for real: Stage 2's reference-slice assertion, the reasoning sub-pass that derives *where* the car goes (§2 "add's placement sub-step"), and the post-hoc DAAM mask refinement. Watch the `[stage2]` and `[add]` log lines.

In [ ]:
!python run_incremental_edit.py add \
    --project-dir {PROJECT} \
    --object car_1 --noun car \
    --prompt "a red sports car parked in the driveway" \
    --seed 42 --steps 28

In [ ]:
show_latest()

In [ ]:
Image.open(f"{PROJECT}/masks/car_1.png")

## 6. `attribute` — recolor the same car

Reuses `car_1`'s stored mask from step 5 — no re-derivation. Should be the same car, different color, background untouched. Check the printed `Preservation gap` — should be positive (PASS).

In [ ]:
!python run_incremental_edit.py attribute \
    --project-dir {PROJECT} \
    --object car_1 --prompt "make the car blue" \
    --seed 42 --steps 28

In [ ]:
show_latest()

## 7. `replace` — swap the car for a bicycle

In [ ]:
!python run_incremental_edit.py replace \
    --project-dir {PROJECT} \
    --object car_1 --new-object cycle_1 --noun cycle \
    --prompt "a bicycle parked in the same spot" \
    --seed 42 --steps 28

In [ ]:
show_latest()

## 8. `part-edit` — change just the bicycle's tire

Exercises the intersection-with-parent-mask logic (`mask_ops.zone_masks` shell zone) — the part noun 'tire' is scoped inside `cycle_1`'s mask so it can't latch onto anything else in frame.

In [ ]:
!python run_incremental_edit.py part-edit \
    --project-dir {PROJECT} \
    --object cycle_1 --part tire_1 --part-noun tire \
    --prompt "a spoked alloy wheel" \
    --seed 42 --steps 28

In [ ]:
show_latest()

## 9. `remove` — take the bicycle back out

No `--fill-prompt` given, so it falls back to revision 0's own prompt (the empty-driveway description) as the background-completion target.

In [ ]:
!python run_incremental_edit.py remove \
    --project-dir {PROJECT} \
    --object cycle_1 \
    --seed 42 --steps 28

In [ ]:
show_latest()

## 10. Inspect drift across the whole chain

`step_psnr` (vs. immediate parent) and `cumulative_psnr` (vs. revision 0) for every revision, per `pipelineInc.md` §6.

In [ ]:
import json, glob

for path in sorted(glob.glob(f"{PROJECT}/runs/rev*_summary.json"), key=lambda p: int(p.split('rev')[1].split('_')[0])):
    with open(path) as f:
        s = json.load(f)
    print(path, "step_psnr=", s.get("step_psnr"), " cumulative_psnr=", s.get("cumulative_psnr"),
          " preservation_gap=", s.get("preservation_gap"))

## If something breaks

- **`[stage2] assertion could not run: ...`** — `assert_reference_slice()` in `kontext_injection.py` guessed the diffusers internal method name (`pipe._encode_vae_image`). Run the cell below to dump the pipeline's actual public/internal API surface, then paste both the error and this dump back for a fix.
- **Suppressed edit** (background perfect, nothing changed) — lower `--inject-cutoff-frac` and/or `--inject-strength` (`pipelineInc.md` §3).
- **Leaking edit** (background drifting) — raise those same two knobs, or the mask is wrong; try `--vital-layers all` (§4's TIER_A-transfer caveat).
- **`AttributeError` inside the attention processor** (`to_q`/`add_q_proj`/`norm_added_k`) — projection names shifted between diffusers versions; the top-level error handler in `run_incremental_edit.py` already prints `attn_processors` keys to help re-derive them.

In [ ]:
# Only needed if Stage 2 failed above.
import kontext_injection as ki
import torch
pipe = ki.load_kontext_pipeline(device="cuda")
print(sorted(m for m in dir(pipe) if 'image' in m.lower() or 'encode' in m.lower() or 'vae' in m.lower()))
print(list(pipe.transformer.attn_processors.keys())[:5])